# LANGFENS — RQ1: TRN-04 T3 Training on Kaggle Tesla T4
Combined Intervention: Class-Conditioned Focal Loss, 4D Waveform Acoustic Evidence Enabled

Private execution notebook under CC BY-NC 4.0.
Executes strictly on authorized remote GPU hardware (Tesla T4) under CUDA FP16 AMP.
Strict fallback prohibition: Never fall back to local CPU/GPU training.
Fixed budget: exactly 3,600 updates.


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys, os
from pathlib import Path
subprocess.run(["nvidia-smi"], check=True)
import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device: {torch.cuda.get_device_name(0)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), f"Expected Tesla T4, got {torch.cuda.get_device_name(0)}"


In [ ]:
# Phase 2: Credentials Resolution (Staged Token & Kaggle Secrets)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not token:
    token_candidates = list(Path("/kaggle/input").rglob("*token*"))
    for tc in token_candidates:
        if tc.is_file():
            t_text = tc.read_text().strip()
            if t_text.startswith("hf_"):
                token = t_text
                break
if token:
    os.environ["HF_TOKEN"] = token
    print(f"[AUTH] Verified HF_TOKEN presence (len={len(token)})")
else:
    print("[WARN] HF_TOKEN not found; local checkpointing to /kaggle/working will proceed.")


In [ ]:
# Phase 3: Setup Repository Code
repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)], check=True)
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)
subprocess.run(["git", "checkout", "review/rq1-trn-04-factorial-training"], cwd=repo_dir, check=True)
head_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
print(f"[GIT] Checked out HEAD: {head_sha}")
sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))
os.environ["PYTHONPATH"] = f"{repo_dir}/src:{repo_dir}"


In [ ]:
# Phase 4: Execute Factorial Training for Arm T3
import hashlib
archive = None
for name in ["authorized_RQ1-TRN-02A_fold1_train_val_only.bin", "authorized_RQ1-TRN-02A_fold1_train_val_only.zip"]:
    matches = list(Path("/kaggle/input").rglob(name))
    if matches:
        archive = matches[0]
        break
assert archive and archive.is_file(), "FAIL-CLOSED: Authorized archive not found in /kaggle/input!"
print(f"[DATASET] Found authorized archive: {archive} ({archive.stat().st_size:,} bytes)")
hasher = hashlib.sha256()
with open(archive, "rb") as f:
    while chunk := f.read(1024*1024*8):
        hasher.update(chunk)
actual_sha = hasher.hexdigest()
EXPECTED_SHA = "b750eb3bf3971afccd9ab9058a64fa6b3f5781399b9ef0f1294d29bf580784a9"
assert actual_sha == EXPECTED_SHA, f"Archive SHA mismatch: {actual_sha} != {EXPECTED_SHA}"
print("[DATASET] SHA-256 Verified authentic!")

arm_name = "T3"
output_dir = Path("/kaggle/working/checkpoints/RQ1-TRN-04/t3")
output_dir.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable,
    str(repo_dir / "audit_scripts/train_trn04_factorial.py"),
    "--arm", arm_name,
    "--split-manifest", str(repo_dir / "manifests/RQ1-SPLIT-01/l2arctic_4fold_splits.json"),
    "--zip-path", str(archive),
    "--output-dir", str(output_dir),
    "--seed", "1337",
    "--batch-size", "2",
    "--gradient-accumulation-steps", "2",
    "--warmup-updates", "100",
    "--max-updates", "3600",
    "--lr", "0.001",
    "--weight-decay", "0.01",
    "--gradient-clip", "1.0",
    "--max-epochs", "25",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
    "--remote-mirror-backend", "huggingface"
]
print("=" * 80)
print(f"[LAUNCH] Executing Training for Arm {arm_name}:")
print("  " + " ".join(train_cmd))
print("=" * 80)
subprocess.run(train_cmd, check=True)
print(f"[LAUNCH] Training completed successfully for Arm {arm_name}.")


In [ ]:
# Phase 5: Durability Verification & Local Artifact Inventory
import json
out_dir = Path("/kaggle/working/checkpoints/RQ1-TRN-04/t3")
print("=" * 80)
print(f"[ARTIFACTS] Final artifacts in {out_dir}:")
for f in sorted(out_dir.glob("*")):
    sha = hashlib.sha256(f.read_bytes()).hexdigest() if f.is_file() else "DIR"
    print(f"  {f.name}: {f.stat().st_size:,} bytes | SHA-256: {sha}")
summary_file = out_dir / "training_summary.json"
if summary_file.is_file():
    print("[SUMMARY] Receipt content:")
    print(summary_file.read_text())
print("=" * 80)
